In [ ]:
#Install requirements
%pip install -r "../requirements.txt"

In [ ]:
#Import required libraries
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns
import scipy
import sys
import glob
from sklearn.naive_bayes import GaussianNB

In [ ]:
#Add the src to the path
sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
#Path for the Data
if not os.path.exists("../data/students_dataset.csv"):
    print("Data Path does not exist. Expected in `../data/students_dataset.csv`")
    exit

data = pd.read_csv(os.path.join("../data/students_dataset.csv"))

In [ ]:
#Import the nested cross-validation class
from src.final_model_CV import CrossValidation
from src.final_model_CV import get_models_statistics
from src.final_model_CV import plot_metrics

In [ ]:
#Run a  final cross validation (5-fold CV)
selected_feat = ["cp", "thal", "ca", "exang", "slope", "thalach", "oldpeak"]
X_final_df = data[selected_feat + ['num']]

estimator = [('Gaussian_Naive_Bayes', GaussianNB())]
parameter_space = {'Gaussian_Naive_Bayes': lambda trial:{'var_smoothing' : trial.suggest_float('var_smoothing', 1e-11, 1e-7, log=True)}}

final_cv = CrossValidation(data=X_final_df, estimators=estimator, parameter_space=parameter_space, n_folds=5, seed=42)
final_cv.runfinalcv()

In [ ]:
#Save the model
final_model = final_cv.save_model('Gaussian_Naive_Bayes', filename="../models/GNB_final.pkl")

In [ ]:
#Get the statistics for the model and plot the selected metrics
model_stats = get_models_statistics(final_cv.results, output_filename = "../data/Task5/GNB_final_metrics_report.csv")
plot_metrics(model_stats, filename = "../figures/Task5/GNB_final_metrics.png")

In [ ]:
#Generate SHAP values
shap_explanation = final_cv.generate_SHAP(model_path="../models/GNB_final.pkl", filename="../data/Task5/GNB_shap.csv")
